In [ ]:
import os
import cv2
import numpy as np

def preprocess_image(image):

    # Step 1: Convert RGB image to YCbCr color space
    ycbcr_image = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)

    # Step 2: Skin detection using YCrCb range
    skin_mask = cv2.inRange(
        ycbcr_image,
        (0, 133, 77),
        (255, 173, 127)
    )

    # Step 3: Convert to binary image
    _, binary_image = cv2.threshold(
        skin_mask,
        0,
        255,
        cv2.THRESH_BINARY
    )

    # Step 4: Morphological closing
    kernel = np.ones((5,5), np.uint8)

    closing = cv2.morphologyEx(
        binary_image,
        cv2.MORPH_CLOSE,
        kernel
    )

    # Step 5: Extract ear region
    ear_color_image = cv2.bitwise_and(
        image,
        image,
        mask=closing
    )

    return ear_color_image


# Load images with filenames
def load_images_from_folder(folder):

    images = []

    for filename in os.listdir(folder):

        img_path = os.path.join(folder, filename)

        if os.path.isfile(img_path):

            img = cv2.imread(img_path)

            if img is not None:

                images.append((filename, img))

    return images


# Input and output folders
input_folder = '.'
output_folder = 'output_images'


# Create output folder
if not os.path.exists(output_folder):
    os.makedirs(output_folder)


# Load images
input_images = load_images_from_folder(input_folder)


# Process and save images
for filename, img in input_images:

    output_img = preprocess_image(img)

    output_path = os.path.join(output_folder, filename)

    cv2.imwrite(output_path, output_img)

    print(f"Processed: {filename}")